Desc: This script reads a Critical_Distance file, extracts the information, decide number of runs for each scenario, and prints out the input to be given in OMNeT++. <br>
Date: 14/02/2024

In [ ]:
'''
PSEUDOCODE
1. Read Crit Dist File
2. Filter for each MCS (repeat the entire process for each different MCS)
3. For each USI 
4.   Filter out the heights where the crit dists are zero for any link. (Justification: The communication is not considered reliable unless all links reliable)
5.   Count the number of heights remaining.
6.   Split the target number of samples over no. of combinations of valid height and UAV speed (num_samples = num_target_samples / (no. remaining heights * no. UAV speed))
7.   For each UAV Speed
8.       For each remaining height
9.           Store the num_samples, critical distance (choose minimum among the three links), uav speed, height and usi 
10.Create the DF of parameters for each simulation case
11.Print the OMNeT++ line to be inputted in the .ini file.
'''

## For OCSVM Training Data. Based on D_max

In [4]:
# NOTE: THIS ASSUMES THROUGHPUT TIME WINDOW OF 1s and THROUGHPUT STRIDE OF 0.1s

import pandas as pd
import math

TARGET_NUM_SAMPLES = 100000 # The target number of samples to train each OCSVM model
UAV_SPEEDS = [6, 16, 26] # List the possible UAV speeds to be simulated
MCS_INDEX = 7 # Repeat for each MCS_INDEX

'''1. Read Crit Dist File'''
CRIT_DIST_FILE = "/home/research-student/omnet-fanet/data-processing-scripts/ocsvm_critical_distances/Dmax_RelTh99_Simulated_Reliability_v3_10e5.csv"
crit_dist_df = pd.read_csv(CRIT_DIST_FILE)

'''2. Filter for each MCS (repeat the entire process for each different MCS)'''
crit_dist_df = crit_dist_df.loc[crit_dist_df["MCS_Index"] == MCS_INDEX]

'''3. For each USI'''
sim_scenario = []
for usi in crit_dist_df["USI"].unique():
    '''4. Filter out the heights where the crit dists are zero for any link.'''
    valid_height = []
    dmax_list = [] 
    usi_crit_dist_df = crit_dist_df.loc[crit_dist_df["USI"] == usi]
    for height in usi_crit_dist_df["Height"].unique():
        h_usi_crit_dist_df = usi_crit_dist_df.loc[usi_crit_dist_df["Height"] == height]
        if h_usi_crit_dist_df["D_max"].values[0] > 0:
            valid_height.append(height)
            dmax_list.append(h_usi_crit_dist_df["D_max"].values[0])
    '''6. Split the target number of samples over no. of combinations of valid height and UAV speed'''
    if len(valid_height) > 0:
        for speed in UAV_SPEEDS:
            for height, dmax in zip(valid_height, dmax_list):
                # Dmax must be at least 1.1 times the speed (else it is insufficient to collect measured throughput samples)
                if ((dmax/speed) > 1.1): 
                    sim_scenario.append({"MCS_Index": MCS_INDEX, "USI": usi, "Height": height, "UAV_Speed": speed, "D_max": dmax})
                else:
                    print({"MCS_Index": MCS_INDEX, "USI": usi, "Height": height, "UAV_Speed": speed, "D_max": dmax})

sim_scenario_df = pd.DataFrame(sim_scenario)
sim_scenario_df.sort_values(by=["UAV_Speed", "USI", "Height"], ignore_index=True, inplace=True)

# Calc number of samples needed for each simulation
for usi in crit_dist_df["USI"].unique():
    usi_sim_scenario_df = sim_scenario_df.loc[(sim_scenario_df["USI"] == usi)]
    if not usi_sim_scenario_df.empty:
        num_samples = math.ceil(TARGET_NUM_SAMPLES/(len(usi_sim_scenario_df)))
        sim_scenario_df.loc[(sim_scenario_df["USI"] == usi), "Num_Samples"] = num_samples

sim_scenario_df["Num_Runs"] = sim_scenario_df["Num_Samples"] / ((sim_scenario_df["D_max"] / sim_scenario_df["UAV_Speed"] - 1) * 10)
sim_scenario_df["Num_Runs"] = sim_scenario_df["Num_Runs"].apply(math.ceil)

print("Total Number of Runs: {}".format(sim_scenario_df["Num_Runs"].sum()))

print("UAV_Speed:")
print(str(sim_scenario_df["UAV_Speed"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Num_Samples:")
print(str(sim_scenario_df["Num_Samples"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("D_max:")
print(str(sim_scenario_df["D_max"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Height:")
print(str(sim_scenario_df["Height"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("USI:")
print(str(sim_scenario_df["USI"].values).replace(". ",",").replace(".7 ",".7,").replace(".\n",", \\\n").replace("7\n","7, \\\n"))

# sim_scenario_df.to_csv("DJISpark_Measured_Throughput_MCS_{}_Dataset_Details.csv".format(MCS_INDEX))

{'MCS_Index': 7, 'USI': 20.0, 'Height': 150, 'UAV_Speed': 26, 'D_max': 20.0}
Total Number of Runs: 17408
UAV_Speed:
[ 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,16,16,16, \
 16,16,16,16,16,16,16,16,16,16,16,16,26,26,26,26,26,26, \
 26,26,26,26,26,26,26,26.]
Num_Samples:
[11112,11112,11112, 9091, 9091, 9091, 9091, 8334, 8334, 8334, \
  8334, 8334, 8334, 8334, 8334,11112,11112,11112, 9091, 9091, \
  9091, 9091, 8334, 8334, 8334, 8334, 8334, 8334, 8334, 8334, \
 11112,11112,11112, 9091, 9091, 9091, 8334, 8334, 8334, 8334, \
  8334, 8334, 8334, 8334.]
D_max:
[ 80, 80, 60, 90, 90, 70, 20,100, 90, 80, 40,100, 90, 80, \
  40, 80, 80, 60, 90, 90, 70, 20,100, 90, 80, 40,100, 90, \
  80, 40, 80, 80, 60, 90, 90, 70,100, 90, 80, 40,100, 90, \
  80, 40.]
Height:
[ 60, 90,120, 60, 90,120,150, 60, 90,120,150, 60, 90,120, \
 150, 60, 90,120, 60, 90,120,150, 60, 90,120,150, 60, 90, \
 120,150, 60, 90,120, 60, 90,120, 60, 90,120,150, 60, 90, \
 120,150.]
USI:
[ 10,  10,  10,  20,  20,  20,  20,  66.7, 

## For Waypoint Mode (Only Broadcast) Dataset to Find Tau_R

In [4]:
# NOTE: THIS ASSUMES THROUGHPUT TIME WINDOW OF 1s and THROUGHPUT STRIDE OF 1s for Throughput-based Retransmission

import pandas as pd
import math
import numpy as np

def get_mcs_from_index(df_in):
    '''
    Gets the MCS index based on modulation and bitrate column of the df_in
    '''
    df = df_in.copy()
    df["Modulation"] = ''
    df["Bitrate"] = np.nan
    df.loc[(df["MCS_Index"] == 0), "Modulation"] = "BPSK" # MCS Index 0
    df.loc[(df["MCS_Index"] == 0), "Bitrate"] = 6.5 # MCS Index 0
    df.loc[(df["MCS_Index"] == 1), "Modulation"] = "QPSK" # MCS Index 0
    df.loc[(df["MCS_Index"] == 1), "Bitrate"] = 13 # MCS Index 0
    df.loc[(df["MCS_Index"] == 2), "Modulation"] = "QPSK" # MCS Index 0
    df.loc[(df["MCS_Index"] == 2), "Bitrate"] = 19.5 # MCS Index 0
    df.loc[(df["MCS_Index"] == 3), "Modulation"] = "QAM-16" # MCS Index 0
    df.loc[(df["MCS_Index"] == 3), "Bitrate"] = 26 # MCS Index 0
    df.loc[(df["MCS_Index"] == 4), "Modulation"] = "QAM-16" # MCS Index 0
    df.loc[(df["MCS_Index"] == 4), "Bitrate"] = 39 # MCS Index 0
    df.loc[(df["MCS_Index"] == 5), "Modulation"] = "QAM-64" # MCS Index 0
    df.loc[(df["MCS_Index"] == 5), "Bitrate"] = 52 # MCS Index 0
    df.loc[(df["MCS_Index"] == 6), "Modulation"] = "QAM-64" # MCS Index 0
    df.loc[(df["MCS_Index"] == 6), "Bitrate"] = 58.5 # MCS Index 0
    df.loc[(df["MCS_Index"] == 7), "Modulation"] = "QAM-64" # MCS Index 0
    df.loc[(df["MCS_Index"] == 7), "Bitrate"] = 65 # MCS Index 0

    return df

TARGET_NUM_SAMPLES = 1000000 # The highest number of samples for tau_R (for convergence)
UAV_SPEEDS = [6, 16, 26] # List the possible UAV speeds to be simulated

'''1. Read Crit Dist File'''
CRIT_DIST_FILE = "/home/research-student/omnet-fanet/data-processing-scripts/ocsvm_critical_distances/Dmax_RelTh99_Simulated_Reliability_v3_10e5.csv"
crit_dist_df = pd.read_csv(CRIT_DIST_FILE)

'''2. Filter for USI of 100 ms '''
crit_dist_df = crit_dist_df.loc[crit_dist_df["USI"] == 100.0]

'''3. For each MCS'''
sim_scenario = []
for mcs_index in crit_dist_df["MCS_Index"].unique():
    '''4. Filter out the heights where the crit dists are zero for any link.'''
    valid_height = []
    dmax_list = [] 
    mcs_crit_dist_df = crit_dist_df.loc[crit_dist_df["MCS_Index"] == mcs_index]
    for height in mcs_crit_dist_df["Height"].unique():
        h_mcs_crit_dist_df = mcs_crit_dist_df.loc[mcs_crit_dist_df["Height"] == height]
        if h_mcs_crit_dist_df["D_max"].values[0] > 0:
            valid_height.append(height)
            dmax_list.append(h_mcs_crit_dist_df["D_max"].values[0])
    '''6. Split the target number of samples over no. of combinations of valid height and UAV speed'''
    if len(valid_height) > 0:
        for speed in UAV_SPEEDS:
            for height, dmax in zip(valid_height, dmax_list):
                # Dmax must be at least 1.1 times the speed (else it is insufficient to collect measured throughput samples)
                if ((dmax/speed) > 1.1): 
                    sim_scenario.append({"MCS_Index": mcs_index, "Height": height, "UAV_Speed": speed, "D_max": dmax})
                else:
                    print({"MCS_Index": mcs_index, "Height": height, "UAV_Speed": speed, "D_max": dmax})

sim_scenario_df = pd.DataFrame(sim_scenario)
sim_scenario_df.sort_values(by=["UAV_Speed", "MCS_Index", "Height"], ignore_index=True, inplace=True)
sim_scenario_df = get_mcs_from_index(sim_scenario_df)

# Calc number of samples needed for each simulation
num_samples = math.ceil(TARGET_NUM_SAMPLES/(len(sim_scenario_df)))
print("Num Samples: {}".format(num_samples))

sim_scenario_df["Num_Runs"] = num_samples / ((sim_scenario_df["D_max"] / sim_scenario_df["UAV_Speed"]))
sim_scenario_df["Num_Runs"] = sim_scenario_df["Num_Runs"].apply(math.ceil)

print("Total Number of Runs: {}".format(sim_scenario_df["Num_Runs"].sum()))

print("UAV_Speed:")
print(str(sim_scenario_df["UAV_Speed"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("D_max:")
print(str(sim_scenario_df["D_max"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Height:")
print(str(sim_scenario_df["Height"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Modulation:")
print(str(sim_scenario_df["Modulation"].values).replace("' ","', ").replace("\n",", \\\n"))

print("Bitrate:")
print(str(sim_scenario_df["Bitrate"].values).replace(". ",",").replace(".5 ",".5,").replace(".\n",", \\\n").replace(".5\n",".5, \\\n"))

Num Samples: 7247
Total Number of Runs: 119851
UAV_Speed:
[ 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, \
  6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, \
  6, 6, 6, 6, 6, 6, 6, 6, 6, 6,16,16,16,16,16,16,16,16, \
 16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16, \
 16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16, \
 16,16,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26, \
 26,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26, \
 26,26,26,26,26,26,26,26,26,26,26,26.]
D_max:
[260,320,360,390,410,430,440,450,450,230,270,300,320,330, \
 340,340,340,340,180,210,230,230,230,230,210,200,170,150, \
 160,170,160,150,130, 90, 30,100, 90, 80, 40, 80, 70, 50, \
 100, 90, 80, 40,260,320,360,390,410,430,440,450,450,230, \
 270,300,320,330,340,340,340,340,180,210,230,230,230,230, \
 210,200,170,150,160,170,160,150,130, 90, 30,100, 90, 80, \
  40, 80, 70, 50,100, 90, 80, 40,260,320,360,390,410,430, \
 440,450,450,230,270,300,320,330,340,340,340,340,180,210, \
 230,230,2

# (OLD: For OCSVM Training Dataset) Considers Min HDist Based on DL, UL, and Vid 

In [4]:
# NOTE: THIS ASSUMES THROUGHPUT TIME WINDOW OF 1s and THROUGHPUT STRIDE OF 0.1s

import pandas as pd
import math

TARGET_NUM_SAMPLES = 100000 # The target number of samples to train each OCSVM model
UAV_SPEEDS = [6, 16, 26] # List the possible UAV speeds to be simulated
MCS_INDEX = 7 # Repeat for each MCS_INDEX

'''1. Read Crit Dist File'''
CRIT_DIST_FILE = "/home/research-student/omnet-fanet/data-processing-scripts/ocsvm_critical_distances/Critical_Distances_RelTh99_Simulated_Reliability_v2_10e4.csv"
# CRIT_DIST_FILE = "/home/research-student/omnet-fanet/data-processing-scripts/ocsvm_critical_distances/Critical_Distances_DNN_Predictions_v2.csv"
crit_dist_df = pd.read_csv(CRIT_DIST_FILE)

'''2. Filter for each MCS (repeat the entire process for each different MCS)'''
crit_dist_df = crit_dist_df.loc[crit_dist_df["MCS_Index"] == MCS_INDEX]

'''3. For each USI'''
sim_scenario = []
for usi in crit_dist_df["USI"].unique():
    '''4. Filter out the heights where the crit dists are zero for any link.'''
    valid_height = []
    min_crit_dist = [] # Use this to determine the number of runs to get minimum no. of samples
    max_crit_dist = [] # Use this to determine the h_dist range to travel for each case
    usi_crit_dist_df = crit_dist_df.loc[crit_dist_df["USI"] == usi]
    for height in usi_crit_dist_df["Height"].unique():
        h_usi_crit_dist_df = usi_crit_dist_df.loc[usi_crit_dist_df["Height"] == height]
        if h_usi_crit_dist_df["Critical_Distance"].all():
            valid_height.append(height)
            min_crit_dist.append(h_usi_crit_dist_df["Critical_Distance"].min())
            max_crit_dist.append(h_usi_crit_dist_df["Critical_Distance"].max())
    '''6. Split the target number of samples over no. of combinations of valid height and UAV speed'''
    if len(valid_height) > 0:
        # num_samples = math.ceil(TARGET_NUM_SAMPLES/(len(valid_height) * len(UAV_SPEEDS)))
        for speed in UAV_SPEEDS:
            for height, min_hdist, max_hdist in zip(valid_height, min_crit_dist, max_crit_dist):
                # Minimum crit dist must be at least 1.1 times the speed (else it is insufficient to collect measured throughput samples)
                if ((min_hdist/speed) > 1.1): 
                    sim_scenario.append({"MCS_Index": MCS_INDEX, "USI": usi, "Height": height, "UAV_Speed": speed, "Min_H_Dist_Range": min_hdist, "Max_H_Dist_Range": max_hdist})
                else:
                    print({"MCS_Index": MCS_INDEX, "USI": usi, "Height": height, "UAV_Speed": speed, "Min_H_Dist_Range": min_hdist, "Max_H_Dist_Range": max_hdist})

sim_scenario_df = pd.DataFrame(sim_scenario)
sim_scenario_df.sort_values(by=["UAV_Speed", "USI", "Height"], ignore_index=True, inplace=True)

# Calc number of samples needed for each simulation
for usi in crit_dist_df["USI"].unique():
    usi_sim_scenario_df = sim_scenario_df.loc[(sim_scenario_df["USI"] == usi)]
    if not usi_sim_scenario_df.empty:
        num_samples = math.ceil(TARGET_NUM_SAMPLES/(len(usi_sim_scenario_df)))
        sim_scenario_df.loc[(sim_scenario_df["USI"] == usi), "Num_Samples"] = num_samples

sim_scenario_df["Num_Runs"] = sim_scenario_df["Num_Samples"] / ((sim_scenario_df["Min_H_Dist_Range"] / sim_scenario_df["UAV_Speed"] - 1) * 10)
sim_scenario_df["Num_Runs"] = sim_scenario_df["Num_Runs"].apply(math.ceil)

print("Total Number of Runs: {}".format(sim_scenario_df["Num_Runs"].sum()))

print("UAV_Speed:")
print(str(sim_scenario_df["UAV_Speed"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Num_Samples:")
print(str(sim_scenario_df["Num_Samples"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Max_H_Dist_Range:")
print(str(sim_scenario_df["Max_H_Dist_Range"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Min_H_Dist_Range:")
print(str(sim_scenario_df["Min_H_Dist_Range"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Height:")
print(str(sim_scenario_df["Height"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("USI:")
print(str(sim_scenario_df["USI"].values).replace(". ",",").replace(".7 ",".7,").replace(".\n",", \\\n").replace("7\n","7, \\\n"))

# sim_scenario_df.to_csv("DJISpark_Measured_Throughput_MCS_{}_Dataset_Details.csv".format(MCS_INDEX))

Total Number of Runs: 19576
UAV_Speed:
[ 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6,16,16,16, \
 16,16,16,16,16,16,16,16,16,16,16,16,26,26,26,26,26,26, \
 26,26,26,26,26,26,26,26,26.]
Num_Samples:
[11112,11112,11112, 8334, 8334, 8334, 8334, 8334, 8334, 8334, \
  8334, 8334, 8334, 8334, 8334,11112,11112,11112, 8334, 8334, \
  8334, 8334, 8334, 8334, 8334, 8334, 8334, 8334, 8334, 8334, \
 11112,11112,11112, 8334, 8334, 8334, 8334, 8334, 8334, 8334, \
  8334, 8334, 8334, 8334, 8334.]
Max_H_Dist_Range:
[ 90, 90, 70,100,100, 80, 40,110,100, 90, 60,110,110, 90, \
  60, 90, 90, 70,100,100, 80, 40,110,100, 90, 60,110,110, \
  90, 60, 90, 90, 70,100,100, 80, 40,110,100, 90, 60,110, \
 110, 90, 60.]
Min_H_Dist_Range:
[ 90, 80, 60, 90, 90, 70, 30,100, 90, 80, 40,100,100, 80, \
  40, 90, 80, 60, 90, 90, 70, 30,100, 90, 80, 40,100,100, \
  80, 40, 90, 80, 60, 90, 90, 70, 30,100, 90, 80, 40,100, \
 100, 80, 40.]
Height:
[ 60, 90,120, 60, 90,120,150, 60, 90,120,150, 60, 90,120, \
 150, 60, 90,120, 6

Min HDist Considering DL and Vid only (For BPSK)

In [3]:
# NOTE: THIS ASSUMES THROUGHPUT TIME WINDOW OF 1s and THROUGHPUT STRIDE OF 0.1s

import pandas as pd
import math

TARGET_NUM_SAMPLES = 100000 # The target number of samples to train each OCSVM model
UAV_SPEEDS = [6, 16, 26] # List the possible UAV speeds to be simulated
MCS_INDEX = 0 # Repeat for each MCS_INDEX

'''1. Read Crit Dist File'''
CRIT_DIST_FILE = "/home/research-student/omnet-fanet/data-processing-scripts/ocsvm_critical_distances/Critical_Distances_Simulated_Reliability_v2.csv"
# CRIT_DIST_FILE = "/home/research-student/omnet-fanet/data-processing-scripts/ocsvm_critical_distances/Critical_Distances_DNN_Predictions_v2.csv"
crit_dist_df = pd.read_csv(CRIT_DIST_FILE)

'''2. Filter for each MCS (repeat the entire process for each different MCS)'''
crit_dist_df = crit_dist_df.loc[crit_dist_df["MCS_Index"] == MCS_INDEX]

'''3. For each USI'''
sim_scenario = []
for usi in crit_dist_df["USI"].unique():
    '''4. Filter out the heights where the crit dists are zero for any link.'''
    valid_height = []
    min_crit_dist = [] # Use this to determine the number of runs to get minimum no. of samples
    max_crit_dist = [] # Use this to determine the h_dist range to travel for each case
    usi_crit_dist_df = crit_dist_df.loc[crit_dist_df["USI"] == usi]
    for height in usi_crit_dist_df["Height"].unique():
        h_usi_crit_dist_df = usi_crit_dist_df.loc[(usi_crit_dist_df["Height"] == height) & (usi_crit_dist_df["Link"] != "UL")]
        if h_usi_crit_dist_df["Critical_Distance"].all():
            valid_height.append(height)
            min_crit_dist.append(h_usi_crit_dist_df["Critical_Distance"].min())
            max_crit_dist.append(h_usi_crit_dist_df["Critical_Distance"].max())
    '''6. Split the target number of samples over no. of combinations of valid height and UAV speed'''
    if len(valid_height) > 0:
        # num_samples = math.ceil(TARGET_NUM_SAMPLES/(len(valid_height) * len(UAV_SPEEDS)))
        for speed in UAV_SPEEDS:
            for height, min_hdist, max_hdist in zip(valid_height, min_crit_dist, max_crit_dist):
                # Minimum crit dist must be at least 1.1 times the speed (else it is insufficient to collect measured throughput samples)
                if ((min_hdist/speed) > 1.1): 
                    sim_scenario.append({"MCS_Index": MCS_INDEX, "USI": usi, "Height": height, "UAV_Speed": speed, "Min_H_Dist_Range": min_hdist, "Max_H_Dist_Range": max_hdist})
                else:
                    print({"MCS_Index": MCS_INDEX, "USI": usi, "Height": height, "UAV_Speed": speed, "Min_H_Dist_Range": min_hdist, "Max_H_Dist_Range": max_hdist})

sim_scenario_df = pd.DataFrame(sim_scenario)
sim_scenario_df.sort_values(by=["UAV_Speed", "USI", "Height"], ignore_index=True, inplace=True)

# Calc number of samples needed for each simulation
for usi in crit_dist_df["USI"].unique():
    usi_sim_scenario_df = sim_scenario_df.loc[(sim_scenario_df["USI"] == usi)]
    if not usi_sim_scenario_df.empty:
        num_samples = math.ceil(TARGET_NUM_SAMPLES/(len(usi_sim_scenario_df)))
        sim_scenario_df.loc[(sim_scenario_df["USI"] == usi), "Num_Samples"] = num_samples

sim_scenario_df["Num_Runs"] = sim_scenario_df["Num_Samples"] / ((sim_scenario_df["Min_H_Dist_Range"] / sim_scenario_df["UAV_Speed"] - 1) * 10)
sim_scenario_df["Num_Runs"] = sim_scenario_df["Num_Runs"].apply(math.ceil)

print("Total Number of Runs: {}".format(sim_scenario_df["Num_Runs"].sum()))

print("UAV_Speed:")
print(str(sim_scenario_df["UAV_Speed"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Num_Samples:")
print(str(sim_scenario_df["Num_Samples"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Max_H_Dist_Range:")
print(str(sim_scenario_df["Max_H_Dist_Range"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Min_H_Dist_Range:")
print(str(sim_scenario_df["Min_H_Dist_Range"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("Height:")
print(str(sim_scenario_df["Height"].values.astype("float")).replace(". ",",").replace(".\n",", \\\n"))

print("USI:")
print(str(sim_scenario_df["USI"].values).replace(". ",",").replace(".7 ",".7,").replace(".\n",", \\\n").replace("7\n","7, \\\n"))

# sim_scenario_df.to_csv("DJISpark_Measured_Throughput_MCS_{}_Dataset_Details.csv".format(MCS_INDEX))

Total Number of Runs: 1034
UAV_Speed:
[ 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, \
 16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16,16, \
 26,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26,26.]
Num_Samples:
[3704,3704,3704,3704,3704,3704,3704,3704,3704,3704,3704,3704, \
 3704,3704,3704,3704,3704,3704,3704,3704,3704,3704,3704,3704, \
 3704,3704,3704,3704,3704,3704,3704,3704,3704,3704,3704,3704, \
 3704,3704,3704,3704,3704,3704,3704,3704,3704,3704,3704,3704, \
 3704,3704,3704,3704,3704,3704.]
Max_H_Dist_Range:
[290,350,390,430,460,490,500,520,530,300,370,410,450,490, \
 510,520,540,560,290,350,390,430,460,490,500,520,530,300, \
 370,410,450,490,510,520,540,560,290,350,390,430,460,490, \
 500,520,530,300,370,410,450,490,510,520,540,560.]
Min_H_Dist_Range:
[200,270,310,330,360,360,390,410,380,240,310,320,370,410, \
 420,440,440,460,200,270,310,330,360,360,390,410,380,240, \
 310,320,370,410,420,440,440,460,200,270,310,330,360,360, \
 390,410,380,240,310,320,370,410,420,440